# Masked Diffusion LM：并行去噪、置信解锁与 Infilling

**面试问题：前向 masking、masked loss 和逐轮置信去噪怎样实现，何时会错误固化？**

## 回答主线

用一个可读任务先建立朴素 baseline，再从基础算子实现核心机制，输出中间状态、指标对照和失败修正。断言只在最后保护最关键的不变量；受控小数据用于解释机制，不冒充真实基础模型质量。

## 真实案例

客服需要补全一句模板：“您的退款将在 [MASK] 个工作日内 [MASK]”。不同于左到右生成，Masked Diffusion 可以同时预测两个空位，再按置信度逐轮解锁。案例展示前向噪声、只在 mask 位计算 loss、两轮并行去噪和中间状态；失败案例让错误 token 获得过高置信，说明需要重掩码或 verifier。

### 输入预览：可读模板与候选词表

In [1]:
import numpy as np  # 导入数组运算实现 masked loss 与置信度调度。

vocabulary = ["三", "五", "到账", "取消", "<MASK>"]  # 定义两个空位可能出现的客服词表。
mask_token = "<MASK>"  # 固定遮盖 token 标识。
clean_tokens = ["您的退款将在", "三", "个工作日内", "到账"]  # 构造正确客服模板序列。
masked_tokens = ["您的退款将在", mask_token, "个工作日内", mask_token]  # 同时遮盖数字和动作两个位置。
masked_positions = [1, 3]  # 保存需要监督和去噪的两个位置。
print("干净序列：", clean_tokens)  # 展示训练目标。
print("前向 masking 后：", masked_tokens)  # 展示扩散式模型的噪声状态。
print("需要预测的位置：", masked_positions)  # 明确 loss 只作用于 mask 位。

干净序列： ['您的退款将在', '三', '个工作日内', '到账']
前向 masking 后： ['您的退款将在', '<MASK>', '个工作日内', '<MASK>']
需要预测的位置： [1, 3]


## Baseline 基线：左到右逐 Token 补全

In [2]:
left_to_right_steps = [  # 构造自回归补全必须经历的两个串行步骤。
    ["您的退款将在", "三", "个工作日内", mask_token],  # 第一步先生成数字。
    ["您的退款将在", "三", "个工作日内", "到账"],  # 第二步再生成动作。
]  # 完成自回归状态序列。
print("左到右生成轨迹：")  # 输出 baseline 串行状态。
for step, state in enumerate(left_to_right_steps, start=1):  # 逐步展示每次只能提交一个新 token。
    print(f"step={step} state={state}")  # 显示串行依赖和最终结果。
print(f"Baseline 串行模型调用轮数={len(left_to_right_steps)}")  # 建立并行去噪的调用次数对照。

左到右生成轨迹：
step=1 state=['您的退款将在', '三', '个工作日内', '<MASK>']
step=2 state=['您的退款将在', '三', '个工作日内', '到账']
Baseline 串行模型调用轮数=2


### 核心实现：Masked-only Loss 与置信度解锁

In [3]:
target_indices = {1: vocabulary.index("三"), 3: vocabulary.index("到账")}  # 把两个 mask 位置映射到正确词表索引。
round_logits = {  # 构造每轮对两个 mask 位置的可解释模型 logits。
    1: {1: np.array([3.6, 1.4, -0.5, -1.0, -2.0]), 3: np.array([-0.5, -0.4, 2.0, 1.8, -2.0])},  # 第一轮数字高置信而动作仍有歧义。
    2: {3: np.array([-0.7, -0.6, 4.2, 0.4, -2.0])},  # 数字解锁后第二轮动作变得高置信。
}  # 完成两轮条件去噪 logits。

def softmax(logits):  # 实现数值稳定 softmax。
    shifted = logits - logits.max()  # 减去最大值避免指数溢出。
    values = np.exp(shifted)  # 计算指数权重。
    return values / values.sum()  # 返回概率分布。

def masked_cross_entropy(logits_by_position, targets):  # 只对仍被 mask 的位置计算平均交叉熵。
    losses = []  # 收集每个监督位置的负对数概率。
    for position, logits in logits_by_position.items():  # 遍历当前轮真正预测的位置。
        probabilities = softmax(logits)  # 计算当前位置词表分布。
        losses.append(-np.log(probabilities[targets[position]] + 1e-12))  # 只读取正确 token 的 NLL。
    return float(np.mean(losses))  # 返回 mask 位平均损失。

first_round_loss = masked_cross_entropy(round_logits[1], target_indices)  # 计算第一轮两个 mask 的训练损失。
print(f"第一轮 masked-only loss={first_round_loss:.4f}")  # 展示没有把可见上下文 token 算入 loss。
for position, logits in round_logits[1].items():  # 逐 mask 展示候选概率。
    probabilities = softmax(logits)  # 计算该位置概率。
    print(f"position={position} top={vocabulary[int(np.argmax(probabilities))]} confidence={probabilities.max():.2%} probs={np.round(probabilities, 3)}")  # 展示为何先解锁数字。

第一轮 masked-only loss=0.4150
position=1 top=三 confidence=87.63% probs=[0.876 0.097 0.015 0.009 0.003]
position=3 top=到账 confidence=49.75% probs=[0.041 0.045 0.498 0.407 0.009]


### 两轮并行去噪过程

In [4]:
def denoise(initial_tokens, schedule, confidence_threshold=0.70):  # 实现按轮次和置信度解锁 token 的有界去噪器。
    state = initial_tokens.copy()  # 复制初始 mask 序列避免修改原输入。
    history = [state.copy()]  # 保存每轮状态供学习者观察。
    for round_index in sorted(schedule):  # 按预定去噪轮次执行模型预测。
        candidates = []  # 收集当前轮超过阈值的候选位置。
        for position, logits in schedule[round_index].items():  # 遍历当前轮模型返回的位置分布。
            if state[position] != mask_token:  # 已经解锁的位置不应重复覆盖。
                continue  # 跳过不可变已提交 token。
            probabilities = softmax(logits)  # 计算词表概率。
            token_index = int(np.argmax(probabilities))  # 选择当前最高置信候选。
            candidates.append((float(probabilities[token_index]), position, vocabulary[token_index]))  # 保存置信度、位置和 token。
        for confidence, position, token in sorted(candidates, reverse=True):  # 先处理最可靠候选。
            if confidence >= confidence_threshold:  # 只有超过阈值才从 mask 状态解锁。
                state[position] = token  # 提交当前 token 到共享状态。
        history.append(state.copy())  # 保存本轮完成后的完整序列。
    return state, history  # 返回最终序列和可视化轨迹。

final_tokens, denoise_history = denoise(masked_tokens, round_logits)  # 对客服模板执行两轮置信度去噪。
print("Masked Diffusion 去噪轨迹：")  # 输出并行状态变化。
for step, state in enumerate(denoise_history):  # 逐轮展示哪些 mask 被解锁。
    print(f"round={step} state={state}")  # 显示第一轮解锁数字、第二轮解锁动作。

Masked Diffusion 去噪轨迹：
round=0 state=['您的退款将在', '<MASK>', '个工作日内', '<MASK>']
round=1 state=['您的退款将在', '三', '个工作日内', '<MASK>']
round=2 state=['您的退款将在', '三', '个工作日内', '到账']


## 结果解读：并行能力来自条件结构而非免费一步生成

In [5]:
diffusion_calls = len(round_logits)  # 统计本例实际模型去噪轮数。
correct_positions = sum(final_tokens[position] == clean_tokens[position] for position in masked_positions)  # 统计两个 mask 的恢复正确数。
print("方案             模型轮数  恢复正确位置  最终文本")  # 输出 baseline 与扩散式对照表。
print(f"左到右生成       {len(left_to_right_steps):>8} {len(masked_positions):>12}/{len(masked_positions)}  {''.join(left_to_right_steps[-1])}")  # 展示自回归串行结果。
print(f"置信去噪         {diffusion_calls:>8} {correct_positions:>12}/{len(masked_positions)}  {''.join(final_tokens)}")  # 展示本例两轮去噪结果。
print("解读：多个 mask 可以同轮预测，但条件依赖、阈值和重掩码策略决定需要多少轮；并行不是自动一步完成。")  # 明确扩散语言模型的速度边界。

方案             模型轮数  恢复正确位置  最终文本
左到右生成              2            2/2  您的退款将在三个工作日内到账
置信去噪                2            2/2  您的退款将在三个工作日内到账
解读：多个 mask 可以同轮预测，但条件依赖、阈值和重掩码策略决定需要多少轮；并行不是自动一步完成。


## 失败案例：错误 Token 高置信后被永久固化

In [6]:
bad_schedule = {1: {1: np.array([1.0, 4.5, -0.5, -1.0, -2.0]), 3: round_logits[1][3]}, 2: round_logits[2]}  # 构造第一轮把“五”错误预测为高置信的模型输出。
bad_final, bad_history = denoise(masked_tokens, bad_schedule)  # 使用不可回退提交策略运行错误去噪。
remasked = bad_final.copy()  # 复制错误结果准备 verifier 修正。
if remasked[1] != clean_tokens[1]:  # 用规则或外部 verifier 检测数字与政策不一致。
    remasked[1] = mask_token  # 把可疑高置信 token 重新放回 mask 状态。
fixed_schedule = {2: {1: np.array([4.8, 0.2, -0.5, -1.0, -2.0])}}  # 构造结合政策证据后的修正 logits。
fixed_final, fixed_history = denoise(remasked, fixed_schedule)  # 对被重新遮盖的位置再次去噪。
print("错误高置信轨迹：", bad_history)  # 展示“五”在第一轮被永久固化。
print("Verifier 重掩码后：", fixed_history, "最终=", fixed_final)  # 展示检测、重掩码和修正过程。

错误高置信轨迹： [['您的退款将在', '<MASK>', '个工作日内', '<MASK>'], ['您的退款将在', '五', '个工作日内', '<MASK>'], ['您的退款将在', '五', '个工作日内', '到账']]
Verifier 重掩码后： [['您的退款将在', '<MASK>', '个工作日内', '到账'], ['您的退款将在', '三', '个工作日内', '到账']] 最终= ['您的退款将在', '三', '个工作日内', '到账']


### 生产边界

In [7]:
generation_contract = {"mask_token": mask_token, "rounds": len(round_logits), "confidence_threshold": 0.70, "remask_policy": "verifier-on-policy-fields", "stop_condition": "no-mask-or-budget", "model_revision": "support-mdlm-r1"}  # 构造可复现的扩散生成策略合同。
print("去噪策略合同：", generation_contract)  # 展示模型权重之外还需版本化的推理参数。
print("生产替换点：真实 MDLM 还需时间条件、随机前向噪声日程、训练采样、并行 kernel、校准置信度和开放文本 verifier。")  # 明确静态 logits 教学与真实模型训练的差距。

去噪策略合同： {'mask_token': '<MASK>', 'rounds': 2, 'confidence_threshold': 0.7, 'remask_policy': 'verifier-on-policy-fields', 'stop_condition': 'no-mask-or-budget', 'model_revision': 'support-mdlm-r1'}
生产替换点：真实 MDLM 还需时间条件、随机前向噪声日程、训练采样、并行 kernel、校准置信度和开放文本 verifier。


## 回归测试：只保护 Mask、终止和重掩码

In [8]:
assert len(target_indices) == len(masked_positions)  # 验证 masked loss 的监督集合与真实 mask 位置一致。
assert final_tokens == clean_tokens  # 验证正常两轮去噪恢复完整客服模板。
assert all(token != mask_token for token in final_tokens)  # 验证预算结束时没有遗留未解析 mask。
assert bad_final[1] == "五"  # 验证高置信错误固化的失败探针稳定存在。
assert fixed_final[1] == "三"  # 验证 verifier 重掩码能够修正政策数字。
print("回归测试通过：mask 监督、正常恢复、终止、高置信反例和重掩码修正均成立。")  # 用少量断言总结生成合同。

回归测试通过：mask 监督、正常恢复、终止、高置信反例和重掩码修正均成立。
